In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder , StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression ,Lasso , Ridge
from sklearn.ensemble import (RandomForestRegressor,ExtraTreesRegressor,BaggingRegressor,
                              AdaBoostRegressor ,GradientBoostingRegressor,HistGradientBoostingRegressor
                              , VotingRegressor,StackingRegressor)
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
import joblib

In [2]:
df=pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19.0,female,27.900,0.0,yes,southwest,16884.92400
1,18.0,male,33.770,1.0,no,southeast,1725.55230
2,28.0,male,33.000,3.0,no,southeast,4449.46200
3,33.0,male,22.705,0.0,no,northwest,21984.47061
4,32.0,male,28.880,0.0,no,northwest,3866.85520


In [3]:
df.shape

(1338, 7)

In [4]:
df.columns

Index(['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges'], dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1296 non-null   float64
 1   sex       1329 non-null   object 
 2   bmi       1296 non-null   float64
 3   children  1314 non-null   float64
 4   smoker    1305 non-null   object 
 5   region    1054 non-null   object 
 6   charges   1314 non-null   float64
dtypes: float64(4), object(3)
memory usage: 73.3+ KB


In [6]:
df.describe()

,age,bmi,children,charges
count,1296.000000,1296.000000,1314.000000,1314.000000
mean,39.225309,30.655197,1.092846,13259.233168
std,14.034162,6.117612,1.205684,12136.291497
min,18.000000,15.960000,0.000000,1121.873900
25%,26.000000,26.220000,0.000000,4724.369462
50%,39.000000,30.380000,1.000000,9289.083100
75%,51.000000,34.600000,2.000000,16639.912515
max,64.000000,53.130000,5.000000,63770.428010


In [7]:
for col in df.select_dtypes('object'):
    print(col,df[col].unique(),'\n',"_"*35)

sex ['female' 'male' nan] 
 ___________________________________
smoker ['yes' 'no' nan] 
 ___________________________________
region ['southwest' 'southeast' 'northwest' 'northeast' nan] 
 ___________________________________


# **Handle NULL Values**

In [8]:
df.isna().sum().sort_values(ascending=False)

region      284
age          42
bmi          42
smoker       33
children     24
charges      24
sex           9
dtype: int64

In [9]:
df.isna().sum().sum()

458

In [10]:
df.dropna(subset='charges',axis=0,inplace=True)

In [11]:
df.reset_index(inplace=True,drop=True)

In [12]:
df.shape[0]-df.isna().sum().sum(),df.shape[0]

(916, 1314)

In [13]:
df.duplicated().sum()

1

In [14]:
df.drop_duplicates(inplace=True)
df.reset_index(inplace=True,drop=True)

In [15]:
X=df.drop('charges',axis=1)
y=df['charges']
X_train,X_test,y_train,y_test=train_test_split(X,y ,test_size=0.20, random_state=42)

In [16]:
cat_columns=X.select_dtypes('object').columns
impute=SimpleImputer(missing_values=np.nan, strategy='most_frequent')
X_train[cat_columns]=impute.fit_transform(X_train[cat_columns])
X_test[cat_columns]=impute.transform(X_test[cat_columns])

In [17]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1050 entries, 140 to 1126
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1017 non-null   float64
 1   sex       1050 non-null   object 
 2   bmi       1023 non-null   float64
 3   children  1040 non-null   float64
 4   smoker    1050 non-null   object 
 5   region    1050 non-null   object 
dtypes: float64(3), object(3)
memory usage: 57.4+ KB


In [18]:
num_columns=['age','bmi']
impute=SimpleImputer(missing_values=np.nan, strategy='mean')
X_train[num_columns]=impute.fit_transform(X_train[num_columns])
X_test[num_columns]=impute.transform(X_test[num_columns])

In [19]:
impute=SimpleImputer(missing_values=np.nan, strategy='median')
X_train[['children']]=impute.fit_transform(X_train[['children']])
X_test[['children']]=impute.transform(X_test[['children']])

# **Encoding & Scaling**

In [20]:
ord=OrdinalEncoder()
X_train[cat_columns]=ord.fit_transform(X_train[cat_columns])
X_test[cat_columns]=ord.transform(X_test[cat_columns])

In [21]:
cols=X.select_dtypes('number').columns
scaler=StandardScaler()
X_train[cols]=scaler.fit_transform(X_train[cols])
X_test[cols]=scaler.transform(X_test[cols])

# **Modeling**

In [22]:
models={
    "LinearRegression":LinearRegression(),
    "Lasso":Lasso(),
    "Ridge":Ridge(),
    "RandomForestRegressor":RandomForestRegressor(n_estimators=15,criterion='squared_error',max_depth=10),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "Bagging": BaggingRegressor(estimator=DecisionTreeRegressor(criterion='squared_error',max_depth=10),n_estimators=50, random_state=42),
    "AdaBoost": AdaBoostRegressor(n_estimators=15, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "HistGradientBoosting": HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    "CatBoost": CatBoostRegressor(verbose=0, iterations=100, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}    

In [23]:
accuracy=[]
for model_name , model in models.items():
    model.fit(X_train,y_train)
    MAE_Train = mean_absolute_error(y_train,model.predict(X_train))
    MAE_Test  =  mean_absolute_error(y_test,model.predict(X_test))

    MSE_Train = mean_squared_error(y_train,model.predict(X_train))
    MSE_Test  =  mean_squared_error(y_test,model.predict(X_test))

    RMSE_Train = np.sqrt(MSE_Train)
    RMSE_Test  = np.sqrt(MSE_Test) 

    R_Train = r2_score(y_train,model.predict(X_train))
    R_Test  =  r2_score(y_test,model.predict(X_test))
    
    accuracy.append([MAE_Train,MSE_Train,RMSE_Train,R_Train,MAE_Test,MSE_Test,RMSE_Test,R_Test])
    


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001122 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 1050, number of used features: 6
[LightGBM] [Info] Start training from score 13170.962000


In [24]:
pd.DataFrame(accuracy,columns=["MAE_Train","MSE_Train","RMSE_Train","R_Train","MAE_Test","MSE_Test","RMSE_Test","R_Test"],index=models.keys())

,MAE_Train,MSE_Train,RMSE_Train,R_Train,MAE_Test,MSE_Test,RMSE_Test,R_Test
LinearRegression,4375.940319,3.940929e+07,6277.681815,0.720737,4401.154093,4.261588e+07,6528.083803,0.751109
Lasso,4376.066633,3.940930e+07,6277.683015,0.720737,4401.545975,4.262126e+07,6528.496310,0.751078
Ridge,4383.486512,3.941227e+07,6277.919423,0.720716,4417.787206,4.278307e+07,6540.876721,0.750133
RandomForestRegressor,1460.246152,6.995879e+06,2644.972398,0.950426,3334.527065,3.479582e+07,5898.797925,0.796781
ExtraTrees,31.236897,2.495283e+05,499.528029,0.998232,3313.009560,4.086281e+07,6392.402556,0.761348
Bagging,1445.504130,7.026379e+06,2650.731830,0.950210,2922.219597,2.985257e+07,5463.750690,0.825651
AdaBoost,4821.884096,3.383440e+07,5816.734631,0.760242,5147.334454,3.929420e+07,6268.508691,0.770509
GradientBoosting,2346.997017,1.789175e+07,4229.863543,0.873215,2901.740189,2.749659e+07,5243.719304,0.839411
HistGradientBoosting,1950.088057,1.061534e+07,3258.119449,0.924777,3194.955894,2.982294e+07,5461.038095,0.825824
XGBoost,1391.991380,6.990774e+06,2644.007238,0.950462,2885.507103,2.759865e+07,5253.441396,0.838815


# **Deployment**

In [25]:
model= XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)

In [26]:
model.fit(X_train,y_train)
MAE_Train = mean_absolute_error(y_train,model.predict(X_train))
MAE_Test  =  mean_absolute_error(y_test,model.predict(X_test))

MSE_Train = mean_squared_error(y_train,model.predict(X_train))
MSE_Test  =  mean_squared_error(y_test,model.predict(X_test))

RMSE_Train = np.sqrt(MSE_Train)
RMSE_Test  = np.sqrt(MSE_Test) 

R_Train = r2_score(y_train,model.predict(X_train))
R_Test  =  r2_score(y_test,model.predict(X_test))
    

In [27]:
print(MAE_Train,'\t',MAE_Test,'\n==================\n',MSE_Train,'\t',MSE_Test,'\n====================\n',RMSE_Train,'\t',RMSE_Test,
      '\n==============\n',R_Train,'\t',R_Test)

1391.9913803511163 	 2885.507103318144 
 6990774.272428378 	 27598646.496643458 
 2644.0072375900145 	 5253.4413955657155 
 0.9504618298518099 	 0.8388149721285558


In [32]:
print(model.score(X_train,y_train)*100,"\t",model.score(X_test,y_test)*100)

95.04618298518099 	 83.88149721285558


In [29]:
joblib.dump(model,"Model.pkl")

['Model.pkl']

In [30]:
joblib.dump(scaler,"Scaler.pkl")

['Scaler.pkl']

In [31]:
joblib.dump(ord,"Encoder.pkl")

['Encoder.pkl']